In [55]:
from typing import Tuple

## RGB -> HSV

In [56]:
R = 0
G = 0 
B = 0
scale_coef_hue = 1
scale_coef_sv = 1

In [57]:
def hue_calc(delta: float, C_max: float, R_1: float, G_1: float, B_1: float) -> float:
    if delta == 0:
        return 0.0
    elif C_max == R_1:
        return 60 * (((G_1 - B_1) / delta) % 6)
    elif C_max == G_1:
        return 60 * (((B_1 - R_1) / delta) + 2)
    elif C_max == B_1:
        return 60 * (((R_1 - G_1) / delta) + 4)

In [58]:
def sat_calc_hsv(C_max: float, delta: float) -> float:
    if C_max == 0:
        return 0.0
    return delta / C_max

In [59]:
def rgb_to_hsv(R: float, G: float, B: float, scale_coef_hue: float, scale_coef_sv: float) -> Tuple[float, float, float]:
    R_1 = R / 255
    G_1 = G / 255
    B_1 = B / 255

    C_max = max(R_1, G_1, B_1)
    C_min = min(R_1, G_1, B_1)

    delta = C_max - C_min

    H = hue_calc(delta, C_max, R_1, G_1, B_1)
    H = H * scale_coef_hue

    S = sat_calc_hsv(C_max, delta)
    S = S * scale_coef_sv

    V = C_max * scale_coef_sv

    return H, S, V

In [60]:
print("H, S, V")
print(rgb_to_hsv(R, G, B, scale_coef_hue, scale_coef_sv))

H, S, V
(0.0, 0.0, 0.0)


## HSV -> RGB

In [61]:
H = 60
L = 1
S = 0.5

In [62]:
def pre_colors_calc(C: float, X: float, H: float) -> Tuple[float, float, float]:
    if H >= 360:
        H = H % 360
    if (0 <= H and H < 60):
        return C, X, 0.0
    elif (60 <= H and H < 120):
        return X, C, 0.0
    elif (120 <= H and H < 180):
        return 0.0, C, X
    elif (180 <= H and H < 240):
        return 0.0, X, C
    elif (240 <= H and H < 300):
        return X, 0.0, C
    elif (300 <= H and H < 360):
        return C, 0.0, X

In [63]:
def hsv_to_rgb(H: float, S: float, V: float) -> Tuple[float, float, float]:
    C = V * S
    X = C * (1 - abs((H / 60) % 2 - 1))
    m = V - C

    R_1, G_1, B_1 = pre_colors_calc(C, X, H)

    R = (R_1 + m) * 255
    G = (G_1 + m) * 255
    B = (B_1 + m) * 255

    return R, G, B

In [64]:
print("R, G, B")
print(hsv_to_rgb(H, L, S))

R, G, B
(127.5, 127.5, 0.0)


## RGB -> HLS

In [65]:
R = 0
G = 0
B = 0
scale_coef_hue = 1
scale_coef_ls = 1

In [66]:
def sat_calc_hls(C_max: float, delta: float, L: float) -> float:
    if delta == 0:
        return 0.0
    return delta / (1 - abs(2 * L - 1))

In [67]:
def rgb_to_hls(R: float, G: float, B: float, scale_coef_hue: float, scale_coef_ls: float) -> Tuple[float, float, float]:
    R_1 = R / 255
    G_1 = G / 255
    B_1 = B / 255

    C_max = max(R_1, G_1, B_1)
    C_min = min(R_1, G_1, B_1)

    delta = C_max - C_min

    L = (C_max + C_min) / 2

    H = hue_calc(delta, C_max, R_1, G_1, B_1)

    S = sat_calc_hls(C_max, delta, L)

    H = H * scale_coef_hue
    L = L * scale_coef_ls
    S = S * scale_coef_ls
    
    return H, L, S

In [68]:
print("H, S, L")
print(rgb_to_hls(R, G, B, scale_coef_hue, scale_coef_ls))

H, S, L
(0.0, 0.0, 0.0)


## HLS -> RGB

In [69]:
H = 0
S = 0
L = 0.75

In [70]:
def hls_to_rgb(H: float, L: float, S: float) -> Tuple[float, float, float]:
    C = (1 - abs(2 * L - 1)) * S
    X = C * (1 - abs((H / 60) % 2 - 1))
    m = L - C / 2

    R_1, G_1, B_1 = pre_colors_calc(C, X, H)

    R = (R_1 + m) * 255
    G = (G_1 + m) * 255
    B = (B_1 + m) * 255

    return R, G, B

In [71]:
print("R, G, B")
print(hls_to_rgb(H, L, S))

R, G, B
(191.25, 191.25, 191.25)


## Картинки

In [72]:
import cv2
import numpy as np
from PIL import Image

### RGB -> HSV

PIL

In [73]:
image = Image.open("origins/mononoke.jpg").convert("RGB")
rgb_array = np.array(image)

hsv_array_pil = np.zeros_like(rgb_array)

for i in range(rgb_array.shape[0]):
    for j in range(rgb_array.shape[1]):
        r, g, b = rgb_array[i, j]
        hsv_array_pil[i, j] = rgb_to_hsv(r, g, b, 255 / 360, 255)

Image.fromarray(hsv_array_pil.astype(np.uint8)).save("results/hsv/mononoke_hsv_pil.png")

OpenCV

In [74]:
image = cv2.imread("origins/mononoke.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

hsv_array_cv2 = np.zeros_like(image, dtype=np.uint8)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        r, g, b = image[i, j]
        hsv_array_cv2[i, j] = rgb_to_hsv(r, g, b, 1 / 2, 255)

cv2.imwrite("results/hsv/mononoke_hsv_cv2.png", hsv_array_cv2)

True

## С помощью библиотек

PIL

In [75]:
image = Image.open("origins/mononoke.jpg")

hsv_image = Image.fromarray(np.uint8(np.array(image.convert("HSV"))))

hsv_image.save("results/hsv/mononoke_hsv_lib_pil.png")

OpenCV

In [76]:
image = cv2.imread("origins/mononoke.jpg")

hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

cv2.imwrite("results/hsv/mononoke_hsv_lib_cv2.jpg", hsv_image)

True

### HSV -> RGB

PIL

In [77]:
rgb_array = np.zeros_like(hsv_array_pil)

for i in range(rgb_array.shape[0]):
    for j in range(rgb_array.shape[1]):
        h, s, v = map(float, hsv_array_pil[i, j])
        r, g, b = hsv_to_rgb(h * 360 / 255, s / 255, v / 255)
        rgb_array[i, j] = (r, g, b)

Image.fromarray(rgb_array).save("results/hsv/mononoke_rgb_pil.png")

OpenCV

In [78]:
rgb_array = np.zeros_like(hsv_array_cv2)

for i in range(rgb_array.shape[0]):
    for j in range(rgb_array.shape[1]):
        h, s, v = map(float, hsv_array_cv2[i, j])
        r, g, b = hsv_to_rgb(h * 2, s / 255, v / 255)
        rgb_array[i, j] = (r, g, b)

rgb_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)

cv2.imwrite("results/hsv/mononoke_rgb_cv2.png", rgb_array)

True

### RGB -> HLS

OpenCV

In [79]:
image = cv2.imread("origins/mononoke.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

hls_array = np.zeros_like(image)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        r, g, b = image[i, j]
        hls_array[i, j] = rgb_to_hls(r, g, b, 1 / 2, 255)

cv2.imwrite("results/hls/mononoke_hls_cv2.png", hls_array)

True

## С помощью библиотеки

OpenCV

In [80]:
image = cv2.imread("origins/mononoke.jpg")  
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 

image_hls = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HLS)

cv2.imwrite("results/hls/mononoke_hls_bib_cv2.jpg", image_hls)


True

### HLS -> RGB

OpenCV

In [81]:
rgb_array = np.zeros_like(hls_array)

for i in range(rgb_array.shape[0]):
    for j in range(rgb_array.shape[1]):
        h, l, s = map(float, hls_array[i, j])
        r, g, b = hls_to_rgb(h * 2, l / 255, s / 255)
        rgb_array[i, j] = (r, g, b)

rgb_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)

cv2.imwrite("results/hls/mononoke_rgb_cv2.png", rgb_array)

True